# PART 2: Threshold Tuning

**Input**: Best model from Stage 3

**Goal**: Find optimal classification threshold to maximize F1-Score

**Output**: Optimal threshold for final evaluation

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve, auc
)
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Load Best Model and Data

In [ ]:
# Load best feature set
train_df = pd.read_csv('../data/train_data_best_features.csv')
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"Data shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Load best model
best_model = joblib.load('../models/final_best_model.pkl')
print(f"\nLoaded model: {type(best_model).__name__}")

# Load model info
with open('../results/best_model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"\nModel Performance (from Stage 3):")
print(f"  F1-Score: {model_info['f1_score']:.4f}")
print(f"  ROC-AUC: {model_info['roc_auc']:.4f}")

## Generate Predictions with Cross-Validation

In [ ]:
# Use cross-validation to get out-of-sample predictions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get probability predictions
y_proba = cross_val_predict(best_model, X, y, cv=cv, method='predict_proba')[:, 1]

print(f"Generated predictions for {len(y_proba)} samples")
print(f"Probability range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")
print(f"Mean probability: {y_proba.mean():.4f}")

## Sweep Thresholds

In [ ]:
# Test thresholds from 0.1 to 0.9
thresholds = np.arange(0.1, 0.9, 0.05)

results = []
for threshold in thresholds:
    y_pred = (y_proba >= threshold).astype(int)
    
    f1 = f1_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred)
    
    results.append({
        'threshold': threshold,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    })

results_df = pd.DataFrame(results)

# Find optimal threshold
optimal_idx = results_df['f1_score'].idxmax()
optimal_threshold = results_df.loc[optimal_idx, 'threshold']
optimal_f1 = results_df.loc[optimal_idx, 'f1_score']

print("\n" + "="*60)
print("THRESHOLD SWEEP RESULTS")
print("="*60)
print(results_df.to_string(index=False))

print("\n" + "="*60)
print("OPTIMAL THRESHOLD")
print("="*60)
print(f"Threshold: {optimal_threshold:.2f}")
print(f"F1-Score: {optimal_f1:.4f}")
print(f"Precision: {results_df.loc[optimal_idx, 'precision']:.4f}")
print(f"Recall: {results_df.loc[optimal_idx, 'recall']:.4f}")

## Visualize Threshold Trade-offs

In [ ]:
# Plot F1-Score vs. Threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: F1, Precision, Recall vs. Threshold
axes[0].plot(results_df['threshold'], results_df['f1_score'], 'b-', label='F1-Score', linewidth=2)
axes[0].plot(results_df['threshold'], results_df['precision'], 'g--', label='Precision')
axes[0].plot(results_df['threshold'], results_df['recall'], 'r--', label='Recall')
axes[0].axvline(x=optimal_threshold, color='black', linestyle=':', label=f'Optimal ({optimal_threshold:.2f})')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Metrics vs. Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y, y_proba)
axes[1].plot(recall_curve, precision_curve, 'b-', linewidth=2)
axes[1].scatter(
    [results_df.loc[optimal_idx, 'recall']], 
    [results_df.loc[optimal_idx, 'precision']],
    color='red', s=100, zorder=5, label=f'Optimal (t={optimal_threshold:.2f})'
)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/threshold_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPlot saved to: results/threshold_analysis.png")

## Save Optimal Threshold

In [ ]:
# Save threshold results
threshold_info = {
    'optimal_threshold': float(optimal_threshold),
    'f1_score': float(optimal_f1),
    'precision': float(results_df.loc[optimal_idx, 'precision']),
    'recall': float(results_df.loc[optimal_idx, 'recall']),
    'default_threshold': 0.5,
    'default_f1': float(results_df[results_df['threshold'] == 0.50]['f1_score'].values[0]) if 0.50 in results_df['threshold'].values else None
}

with open('../results/optimal_threshold.json', 'w') as f:
    json.dump(threshold_info, f, indent=2)

print("Optimal threshold saved to: results/optimal_threshold.json")

# Save full results
results_df.to_csv('../results/threshold_sweep_results.csv', index=False)
print("Full threshold sweep saved to: results/threshold_sweep_results.csv")

## Summary

**THRESHOLD TUNING COMPLETE!** ✅

**Key Findings**:
- Optimal threshold may differ from default 0.5
- Precision-Recall trade-off visualized
- Ready for final holdout evaluation

**Next Step**: Run `final_evaluation.ipynb` ⚠️ ONE TIME ONLY!

**Outputs Created**:
- `results/optimal_threshold.json` - Threshold to use
- `results/threshold_sweep_results.csv` - All tested thresholds
- `results/threshold_analysis.png` - Visualization